In [ ]:
!pip install -q langchain-google-genai google-generativeai sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.7 MB/s eta 0:00:00


In [ ]:
import os
import shutil
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings  # ← Gratis, local
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI  # ← Gemini

print("✅ Todas las librerías importadas correctamente")

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [ ]:
!pip install -q langchain langchain-community pypdf pandas chromadb
!pip install -q langchain-google-genai google-generativeai
!pip install -q sentence-transformers

In [4]:
import os
import shutil

# Crea la carpeta data
!mkdir -p data

# Mueve todos los PDFs a la carpeta data
for archivo in os.listdir('/content'):
    if archivo.endswith('.pdf'):
        shutil.move(f'/content/{archivo}', f'data/{archivo}')
        print(f"✅ Movido: {archivo}")

print("\n📂 Archivos en data/:")
!ls data/

✅ Movido: Manual de Garantía de Productos de BimBam Buy.pdf
✅ Movido: Programa de Afiliados de BimBam Buy.pdf
✅ Movido: Política de Reembolsos y Devoluciones de BimBam Buy.pdf
✅ Movido: Preguntas_Frecuentes_sobre_Métodos_de_Pago_de_BimBam_Buy.pdf
✅ Movido: Guía de Tiempos y Costos de Envío de BimBam Buy.pdf

📂 Archivos en data/:
'Guía de Tiempos y Costos de Envío de BimBam Buy.pdf'
'Manual de Garantía de Productos de BimBam Buy.pdf'
'Política de Reembolsos y Devoluciones de BimBam Buy.pdf'
 Preguntas_Frecuentes_sobre_Métodos_de_Pago_de_BimBam_Buy.pdf
'Programa de Afiliados de BimBam Buy.pdf'


In [6]:
def cargar_documentos(ruta="data/"):
    """Carga todos los PDFs y los divide en chunks"""
    documentos = []

    for archivo in os.listdir(ruta):
        if archivo.endswith('.pdf'):
            print(f"📄 Cargando: {archivo}")
            loader = PyPDFLoader(os.path.join(ruta, archivo))
            documentos.extend(loader.load())

    # Divide el texto en chunks pequeños para mejor búsqueda
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # Cada chunk tiene ~1000 caracteres
        chunk_overlap=200     # Se solapan 200 caracteres entre chunks
    )
    chunks = text_splitter.split_documents(documentos)

    print(f"\n✅ Total de documentos cargados: {len(documentos)}")
    print(f"✅ Total de chunks generados: {len(chunks)}")
    return chunks

# Ejecuta la función
chunks = cargar_documentos()

📄 Cargando: Manual de Garantía de Productos de BimBam Buy.pdf


NameError: name 'PyPDFLoader' is not defined

In [7]:
def crear_vectorstore(chunks, persist_directory="./chroma_db"):
    """Crea la base de vectores con embeddings gratuitos de HuggingFace"""
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    vectorstore.persist()

    print("✅ Base de vectores creada y guardada")
    print(f"📁 Ubicación: {persist_directory}")
    return vectorstore

# Ejecuta la función
vectorstore = crear_vectorstore(chunks)

NameError: name 'chunks' is not defined

In [8]:
def crear_agente(vectorstore):
    """Crea el agente de QA con RAG usando Gemini (gratis)"""
    # Usa Gemini Flash - modelo gratuito y rápido
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0.3,
        google_api_key=os.environ["GOOGLE_API_KEY"]
    )

    # Crea la cadena de Retrieval QA
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True
    )

    return qa_chain

# Crea el agente
agente = crear_agente(vectorstore)
print("🤖 Agente BimBam Buy listo para responder preguntas!")
print("   Modelo: Gemini 1.5 Flash")
print("   Base de conocimiento: 5 documentos de BimBam Buy")

NameError: name 'vectorstore' is not defined

In [9]:
def preguntar(pregunta):
    """Hace una pregunta al agente y muestra la respuesta con fuentes"""
    print("=" * 70)
    print(f"❓ PREGUNTA: {pregunta}")
    print("=" * 70)

    respuesta = agente({"query": pregunta})

    print(f"\n💬 RESPUESTA:\n{respuesta['result']}\n")
    print("-" * 70)
    print("📚 FUENTES UTILIZADAS:")
    for i, doc in enumerate(respuesta['source_documents'], 1):
        fuente = os.path.basename(doc.metadata.get('source', 'Desconocida'))
        print(f"\n  Fuente {i}: {fuente}")
        print(f"  {doc.page_content[:250]}...")

    return respuesta

# 🧪 Primera prueba
preguntar("¿Cuál es la política de reembolsos de BimBam Buy?")

❓ PREGUNTA: ¿Cuál es la política de reembolsos de BimBam Buy?


NameError: name 'agente' is not defined

In [10]:
# 🧪 Más preguntas de prueba

preguntar("¿Cómo funciona el programa de afiliados de BimBam Buy?")

preguntar("¿Cuáles son los métodos de pago aceptados?")

preguntar("¿Cuánto tarda un envío en llegar?")

preguntar("¿Qué cubre la garantía de los productos?")

preguntar("¿Puedo devolver un producto si no me gustó?")

❓ PREGUNTA: ¿Cómo funciona el programa de afiliados de BimBam Buy?


NameError: name 'agente' is not defined

In [12]:
from google.colab import files

# Descarga el notebook actual
files.download('bim_bam_buy_agent.ipynb')

FileNotFoundError: Cannot find file: bim_bam_buy_agent.ipynb